In [0]:
!pip install matplotlib  scikit-learn shap 



Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [0]:
df_clean=pd.read_csv("/Volumes/data_warehouse/default/car_price_dataclean/autos_clean.csv",index_col=0)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236338 entries, 1 to 371526
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   price              236338 non-null  int64 
 1   vehicleType        236338 non-null  object
 2   car_age            236338 non-null  int64 
 3   gearbox            236338 non-null  object
 4   powerPS            236338 non-null  int64 
 5   model              236338 non-null  object
 6   kilometer          236338 non-null  int64 
 7   fuelType           236338 non-null  object
 8   brand              236338 non-null  object
 9   notRepairedDamage  236338 non-null  object
dtypes: int64(4), object(6)
memory usage: 19.8+ MB


In [0]:
df_clean["kilometer"]=df_clean["kilometer"].astype("object")

# # lấy 10000 dòng để chạy mô hình
# df_clean=df_clean.sample(n=10000,random_state=42)


In [0]:
#Chuẩn bị dữ liệu:
target="price"
X=df_clean.drop(columns=[target])
y=df_clean[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#preprocessor 
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer=Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("scale",StandardScaler())
])
categorical_transformer=Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,         
        min_frequency=50))
])

#Ráp các pipeline lại với nhau
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    sparse_threshold=0.3   # nếu >30% đầu ra là sparse -> giữ dạng sparse
)

In [0]:
models = {
    "LinearRegression": Pipeline([
        ("prep", preprocessor),
        ("reg", LinearRegression()),
    ]),

    "RandomForest": Pipeline([
        ("prep", preprocessor),
        ("reg", RandomForestRegressor(
            n_estimators=300,      # giảm nhẹ để size nhỏ hơn
            max_depth=24,          # giới hạn độ sâu để file gọn
            n_jobs=-1,
            random_state=42
        )),
    ]),
}

In [0]:
# 2) Huấn luyện & đánh giá
import gc
def evaluate(model, X_tr, y_tr, X_te, y_te, name="Model"):
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)

    mae = round(mean_absolute_error(y_te, pred_te),2)
    rmse = round(mean_squared_error(y_te, pred_te, squared=False),2)
    r2 = round(r2_score(y_te, pred_te),3)

    # lưu kết quả vào dictionary
    df_result_model = {
        "model": name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
        }

    print(f"=== {name} ===")
    print(f"Train R²: {r2_score(y_tr, pred_tr):.3f}")
    print(f"Test  →  MAE: {mae:,} | RMSE: {rmse:,} | R²: {r2}")
    return model,df_result_model
fitted = {}
df_result_model=[]
for name, pipe in models.items():
    fitted[name], df_result = evaluate(pipe, X_train, y_train, X_test, y_test, name=name)
    df_result_model.append(df_result)
    gc.collect()

#lưu kết quả của các mô hình vào một dataframe
df_result_model = pd.DataFrame(df_result_model).sort_values("mae").reset_index(drop=True)
df_result_model


=== LinearRegression ===
Train R²: 0.762
Test  →  MAE: 2,138.33 | RMSE: 3,143.76 | R²: 0.759
=== RandomForest ===
Train R²: 0.962
Test  →  MAE: 1,130.95 | RMSE: 1,896.2 | R²: 0.912


,model,mae,rmse,r2
0,RandomForest,1130.95,1896.20,0.912
1,LinearRegression,2138.33,3143.76,0.759


In [0]:
best_name=df_result_model["model"][0]
print(best_name)
models[best_name]

RandomForest


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['car_age', 'powerPS']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 min_frequency=50))]),
                                                  ['vehicleType', 'gearbox',
                                                   'model', 'kilometer',
                                                   'fuelType', 'brand',
                                                   'notRepairedDamage'])])),
                ('reg',
                 RandomForestRegressor(max_depth=24, n_estimators=300,
                                       n_jobs=-1, random_state=42))])

In [0]:
#chọn mô hình tốt nhất
best_pipe = fitted[best_name]
prep=best_pipe.named_steps["prep"]

final_est = best_pipe.named_steps["reg"]
importances = final_est.feature_importances_

# Lấy tên cột sau transform
num_cols = prep.transformers_[0][2]                      # list numeric gốc
ohe = prep.named_transformers_["cat"].named_steps["onehot"]
cat_cols = prep.transformers_[1][2]                      # list categorical gốc
cat_names = list(ohe.get_feature_names_out(cat_cols))    # tên sau one-hot


feat_names = list(num_cols) + cat_names

fi = pd.DataFrame({"feature_transformed": feat_names,
                   "importance": importances}).sort_values("importance", ascending=False)

print(fi.head(10))

         feature_transformed  importance
0                    car_age    0.566822
1                    powerPS    0.261284
236         kilometer_150000    0.015688
197        model_transporter    0.015140
3    vehicleType_convertible    0.010212
245               brand_audi    0.006445
10         gearbox_automatic    0.006165
264      brand_mercedes_benz    0.004962
286    notRepairedDamage_yes    0.004899
270            brand_porsche    0.003930


In [0]:
print(len(feat_names))

287


**Lưu mô hình**

In [0]:
# import pickle 
# for model_name,pipe in models.items():
#     model_path=f"./models/{model_name}.pkl"
#     with open(model_path, "wb") as file:
#         pickle.dump(pipe, file)

import joblib, os
os.makedirs("./models", exist_ok=True)

# Lưu các mô hình đã fit với nén
for model_name, pipe in fitted.items():
    joblib.dump(pipe, f"./models/{model_name}.joblib", compress=3, protocol=5)

In [0]:
# áp dụng mô hình để dự đoán giá cho file test_car_regester.csv
#đọc file test_car_regester.csv
df_test=pd.read_csv("/Volumes/data_warehouse/default/car_price_dataclean/car_price_test.csv",index_col=0)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19 entries, 1 to 19
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              0 non-null      float64
 1   vehicleType        19 non-null     object 
 2   car_age            19 non-null     int64  
 3   gearbox            19 non-null     object 
 4   powerPS            19 non-null     int64  
 5   model              19 non-null     object 
 6   kilometer          19 non-null     int64  
 7   fuelType           19 non-null     object 
 8   brand              19 non-null     object 
 9   notRepairedDamage  19 non-null     object 
dtypes: float64(1), int64(3), object(6)
memory usage: 1.6+ KB


In [0]:
df_test=df_test.drop(columns=["price"])
df_test["kilometer"]=df_test["kilometer"].astype("object")
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19 entries, 1 to 19
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   vehicleType        19 non-null     object
 1   car_age            19 non-null     int64 
 2   gearbox            19 non-null     object
 3   powerPS            19 non-null     int64 
 4   model              19 non-null     object
 5   kilometer          19 non-null     object
 6   fuelType           19 non-null     object
 7   brand              19 non-null     object
 8   notRepairedDamage  19 non-null     object
dtypes: int64(2), object(7)
memory usage: 1.5+ KB


In [0]:
#dự đoán giá cho file test_car_regester.csv bằng cách gọi mô hình đã lưu

# with open("./models/RandomForest.pkl", "rb") as file:
#     model = pickle.load(file)
# df_test["price"]=model.predict(df_test)

model = joblib.load("./models/RandomForest.joblib")
df_test["price"] = model.predict(df_test)

df_test.info()




<class 'pandas.core.frame.DataFrame'>
Int64Index: 19 entries, 1 to 19
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   vehicleType        19 non-null     object 
 1   car_age            19 non-null     int64  
 2   gearbox            19 non-null     object 
 3   powerPS            19 non-null     int64  
 4   model              19 non-null     object 
 5   kilometer          19 non-null     object 
 6   fuelType           19 non-null     object 
 7   brand              19 non-null     object 
 8   notRepairedDamage  19 non-null     object 
 9   price              19 non-null     float64
dtypes: float64(1), int64(2), object(7)
memory usage: 1.6+ KB


In [0]:
df_test.head(10)

,vehicleType,car_age,gearbox,powerPS,model,kilometer,fuelType,brand,notRepairedDamage,price
FALSE,,,,,,,,,,
1,coupe,2,manual,261,unknown,90000,electric,audi,yes,31738.711746
2,suv,6,automatic,238,grand,150000,diesel,jeep,unknown,23722.657344
3,small car,2,manual,186,golf,60000,petrol,volkswagen,no,19746.131774
4,small car,5,manual,92,fabia,20000,diesel,skoda,no,7889.928331
5,convertible,7,manual,165,2_reihe,70000,petrol,peugeot,no,15727.794210
6,bus/van,3,manual,202,c_max,30000,electric,ford,unknown,20278.542514
7,sedan,11,manual,257,3_reihe,100000,petrol,mazda,no,10271.380266
8,station wagon,5,manual,125,passat,50000,diesel,volkswagen,yes,11161.515569
9,station wagon,10,manual,95,passat,90000,other,volkswagen,no,6670.816740


In [0]:
# Lưu kết quả vào file csv
df_test.to_csv("/Volumes/data_warehouse/default/car_price_dataclean/car_price_test_predict.csv")